## ***IMPORT LIBRERIE***


In [1]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR, FIGURES_DIR, MODELS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob

from models.DeepConvLSTM import DeepConvLSTM, HARDataset 
import optuna
from optuna.pruners import MedianPruner
import optuna.visualization as vis
import train
import torch
import torch.nn as nn
import train_with_cm
import matplotlib.pyplot as plt
#definisco il path da cui leggere i .csv

from utils.log_config import logger
from figures import plot_CM

#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\pdd_data'
logger.debug(path)

2025-05-04 21:26:27.876 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition
2025-05-04 21:26:31,548 - INFO - myapp - Logging configured from C:\codes\HumanActivityRecognition\HumanActivityRecognition\utils\base_config.json
2025-05-04 21:26:39,268 - INFO - myapp - GPU not available, training on CPU.
c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-04 21:26:39,863 - DEBUG - matplotlib - matplotlib data path: c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\matplotlib\mpl-data
2025-05-04 21:26:39,873 - DEBUG - matplotlib - CONFIGDIR=C:\Users\carol\.matplotlib
2025-05-04 21:26:39,902 - DEBUG - matplotlib - interactive is False
2025-05-04 21:26:39,902 - DEBUG - matplotlib - platform is win32
2025-05-04 21:26:40,025 

## ***DATA PREPROCESSING***

*scansiono recording e filtro per righe non nulle*  
*OUTPUT: unico dataframe con tutte le attività non nulle*

In [2]:
#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path
# Definisco il percorso della cartella contenente i CSV

# Nome del file CSV finale
final_csv_path = os.path.join(path, 'df_SPOON_non_null.csv')

# Controllo se il file esiste già
if os.path.exists(final_csv_path):
    logger.debug(f"Il file {final_csv_path} esiste già. Lo sto caricando...")
    df_spoon = pd.read_csv(final_csv_path)
else:
    #trovo tutti i file che corrispondono a "DO" nella cartella path e li stampo a schermo
    files = glob.glob(os.path.join(path, "*_SP*.csv"))
    logger.debug(f"Files: {files}")
    
    kid_spoon, kid_spoon_no_null = [], [] # liste per salvare utenti prima e dopo il merge 
    df_list_spoon = [] # lista vuota per appendere i dataframe con attività non nulla

    for file in files:
        df = pd.read_csv(file)
        logger.debug(f"Original shape: {df.shape}")
        kid_spoon.append(df['kid_id'].unique())
        df = df[df['action_id'] != 0] #filtro le righe con action_id non nullo
        logger.debug(f"Filtered shape: {df.shape}")
        kid_spoon_no_null.append(df['kid_id'].unique())

        logger.debug(f"Columns: {df.columns}")
        logger.debug(f"Action counts:\n{df['action'].value_counts()}")
        logger.debug(f"Toy counts:\n{df['toy_id'].value_counts()}")
        logger.debug("="*50)

    # mi stampo gli utenti prima di fare il merge e dopo il merge
    logger.info(f"Numero di utenti che hanno fatto almeno un azione: {len(kid_spoon_no_null)/len(kid_spoon)}")

    '''
    Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
    al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
    appendo tutte le righe non nulle in un unico dataframe per tutti i file che terminano in .csv nella balltella path
    '''

    for file in os.listdir(path):
        if not file.endswith('.csv'):
            continue
    
        #leggo solo i file che dopo l'undescore ha SP*.csv
        if file.split('_')[-1].startswith('SP') and file.endswith('.csv'): #controllo che il file termini con .csv
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_spoon.append(df_temp)

    df_spoon = pd.concat(df_list_spoon)
    logger.info(f"Shape finale del dataframe: {df_spoon.shape}")
    logger.info(f"Colonne del dataframe: {list(df_spoon.columns)}")
    logger.info(f"Conteggio delle azioni:\n{df_spoon['action'].value_counts()}")

    # salvo il dataframe
    df_spoon.to_csv(os.path.join(path,'df_SPOON_non_null.csv'),index=False)

2025-05-04 21:29:29,697 - DEBUG - myapp - Files: ['C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3002_SP.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3005_SP.csv']
C:\Users\carol\AppData\Local\Temp\ipykernel_30244\4269874133.py:23: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
2025-05-04 21:29:30,691 - DEBUG - myapp - Original shape: (168935, 25)
2025-05-04 21:29:30,706 - DEBUG - myapp - Filtered shape: (1463, 25)
2025-05-04 21:29:30,710 - DEBUG - myapp - Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
2025-05-04 21:29:30,722 - DEBUG - myapp - Action cou

*per ogni attività trovata, salvo un .csv*  
*OUTPUT: un .csv per ogni attività non nulla (df_spoon_action_11.csv, df_spoon_action_19.csv ecc... )*


In [3]:
#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_spoon['action_id'].unique():
    df_action = df_spoon[df_spoon["action_id"] == action_id] #filtro il dataframe in base all'attività
    logger.debug(f"Dimensioni del dataframe df_spoon_action_{action_id} - {df_action.shape}") #log delle dimensioni del dataframe
    logger.info(f"Conteggio delle attività per df_spoon_action_{action_id}") #log del conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_spoon_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    logger.debug(f"Salvato il dataframe df_spoon_action_{action_id}.csv")

2025-05-04 21:29:39,736 - DEBUG - myapp - Dimensioni del dataframe df_spoon_action_41 - (630, 25)
2025-05-04 21:29:39,737 - INFO - myapp - Conteggio delle attività per df_spoon_action_41
2025-05-04 21:29:39,760 - DEBUG - myapp - Salvato il dataframe df_spoon_action_41.csv
2025-05-04 21:29:39,764 - DEBUG - myapp - Dimensioni del dataframe df_spoon_action_10 - (657, 25)
2025-05-04 21:29:39,766 - INFO - myapp - Conteggio delle attività per df_spoon_action_10
2025-05-04 21:29:39,788 - DEBUG - myapp - Salvato il dataframe df_spoon_action_10.csv
2025-05-04 21:29:39,791 - DEBUG - myapp - Dimensioni del dataframe df_spoon_action_11 - (176, 25)
2025-05-04 21:29:39,793 - INFO - myapp - Conteggio delle attività per df_spoon_action_11
2025-05-04 21:29:39,803 - DEBUG - myapp - Salvato il dataframe df_spoon_action_11.csv
2025-05-04 21:29:39,805 - DEBUG - myapp - Dimensioni del dataframe df_spoon_action_4 - (68, 25)
2025-05-04 21:29:39,807 - INFO - myapp - Conteggio delle attività per df_spoon_action

## ***DATA PROCESSING***

*SLIDING WINDOW*  
*OUTPUT: unica matrice con tutte le windows concatenate*

In [4]:
#applico sliding window con la funzion process_csv
nb_sensor_channels = 9
sliding_window_length = 100
sliding_window_step = 50

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al giocattolo spoon
#e poi concateno tutto in un unica x e y 

X, Y = [], []
kid_action_counts = {}

for action_file in [f for f in os.listdir(path) if f.endswith('.csv') and f.split('_')[1] == 'spoon']:
    file_path = os.path.join(path, action_file)
    action = action_file.split('_')[-1].split('.')[0]

    X_windows, Y_windows, kid_id_action_dict = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)

    X.append(X_windows)
    Y.append(Y_windows)


    logger.info(f"Numero  totale di finestre per l'azione {action}:{len(X_windows)}")
    kid_action_counts[f"spoon_action_{action}"] = kid_id_action_dict #aggiungo il dizionario al dizionario principale per tenere traccia del numero di finestre per ogni bambino per ogni azione, ogni spoon_action è una chiave e il valore è un dizionario con il numero di finestre per ogni bambino
    logger.info(f"Contenuto finale di kid_action_counts: {kid_action_counts}") #per veere quante finestre per ogni azione e per ogni bambino sono state elaborte 


# Concateno tutti i dati in un unico array per X e Y
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)


# Stampo le dimensioni di X e Y
logger.info(f"Dimensioni di X finale: {X.shape}")
logger.info(f"Dimensioni di Y finale: {Y.shape}")

2025-05-04 21:30:21,819 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_spoon_action_10.csv
2025-05-04 21:30:21,826 - INFO - myapp - Kid_id: 3002, X_kid shape: (657, 9), Y_kid shape: (657,)
2025-05-04 21:30:21,833 - INFO - myapp - Numero di finestre estratte: 12
2025-05-04 21:30:21,835 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 7
2025-05-04 21:30:21,837 - INFO - myapp - Non è stato applicato padding perché i campioni rimanenti sono troppo pochi.
2025-05-04 21:30:21,840 - INFO - myapp - Numero totale di finestre (dopo padding finale): 12
2025-05-04 21:30:21,842 - DEBUG - myapp - X_windows shape after sliding window: (12, 100, 9)
2025-05-04 21:30:21,843 - DEBUG - myapp - Padding codes: []
2025-05-04 21:30:21,845 - INFO - myapp - Numero di finestre estratte: 12
2025-05-04 21:30:21,847 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 7
2025-05-04 21:30:21,849 - INFO - myapp - Non è stato applicat

>>> Kid_ids: [3002]
>>> Kid_ids: [3002]
>>> Kid_ids: [3005]


2025-05-04 21:30:22,006 - DEBUG - myapp - Padding codes: [1]
2025-05-04 21:30:22,007 - DEBUG - myapp - Y_windows shape after extraction: (1, 1)
2025-05-04 21:30:22,009 - INFO - myapp - Kid_id: 3005, Action_id: 4.0, Action_count: 1
2025-05-04 21:30:22,010 - DEBUG - myapp - Conteggio finale di finestre per ogni bambino per l'azione 4.0: {3005: 1}
2025-05-04 21:30:22,012 - INFO - myapp - Numero  totale di finestre per l'azione 4:1
2025-05-04 21:30:22,015 - INFO - myapp - Contenuto finale di kid_action_counts: {'spoon_action_10': {3002: 12}, 'spoon_action_11': {3002: 3}, 'spoon_action_4': {3005: 1}}
2025-05-04 21:30:22,059 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_spoon_action_41.csv
2025-05-04 21:30:22,063 - INFO - myapp - Kid_id: 3002, X_kid shape: (630, 9), Y_kid shape: (630,)
2025-05-04 21:30:22,065 - INFO - myapp - Numero di finestre estratte: 11
2025-05-04 21:30:22,068 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra com

>>> Kid_ids: [3002]


*SPLIT DATASET IN TRS, VS,TS*  
*OUTPUT: array NumPy di TRS/VS/TS (X e Y)*

*RIASSEGNAZIONE DELLE ETICHETTE*

## ***OPTUNA***

*nella funzione obiettivo: uso train e val*  
*stampo cm di train e val*  
*train finale con dati di test con cm*


